In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import timm

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

C:\Users\abida\anaconda3\envs\deepfake\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = "C:/Users/abida/DEEPFAKE/DeepFakeDataset"      # your dataset folder
IMAGE_SIZE = 384          # your dataset size
BATCH_SIZE = 8            # reduce if GPU memory error
EPOCHS = 15
LR = 5e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", DEVICE)



Using Device: cuda


In [3]:
from torchvision import transforms

IMAGE_SIZE = 384

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# ✅ TEST TRANSFORM (No augmentation!)
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [4]:
train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transform)
test_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Classes:", train_dataset.classes)


Classes: ['fake', 'real']


In [5]:
model = timm.create_model("xception", pretrained=True)

# Replace classifier safely
if hasattr(model, 'fc'):
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)
elif hasattr(model, 'classifier'):
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, 1)

model = model.to(DEVICE)

C:\Users\abida\anaconda3\envs\deepfake\lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to C:\Users\abida/.cache\torch\hub\checkpoints\xception-43020ad28.pth


In [6]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
# loss and optimiser

In [13]:
print("Total Training Images:", len(train_dataset))
print("Total Training Batches:", len(train_loader))

Total Training Images: 135563
Total Training Batches: 16946


In [7]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.delta = delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_acc, model):
        if self.best_score is None:
            self.best_score = val_acc
            self.save_model(model)

        elif val_acc < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping Counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_acc
            self.save_model(model)
            self.counter = 0

    def save_model(self, model):
        torch.save(model.state_dict(), "best_xception_384.pth")
        print("Best model saved!")



In [8]:
# train function
def train_one_epoch(loader):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader):
        images = images.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).int()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc


In [9]:
def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            outputs = model(images)
            preds = (torch.sigmoid(outputs) > 0.5).int()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return acc, precision, recall, f1


In [10]:
print("Freezing backbone...")
for param in model.parameters():
    param.requires_grad = False

if hasattr(model, 'fc'):
    for param in model.fc.parameters():
        param.requires_grad = True
elif hasattr(model, 'classifier'):
    for param in model.classifier.parameters():
        param.requires_grad = True

Freezing backbone...


In [11]:
# training loop
early_stopping = EarlyStopping(patience=5)

for epoch in range(EPOCHS):

    if epoch == 3:
        print("Unfreezing full model...")
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.Adam(model.parameters(), lr=LR)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(train_loader)
    val_acc, val_prec, val_rec, val_f1 = evaluate(val_loader)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")
    print(f"Val Precision: {val_prec:.4f}")
    print(f"Val Recall: {val_rec:.4f}")
    print(f"Val F1: {val_f1:.4f}")

    early_stopping(val_acc, model)

    if early_stopping.early_stop:
        print("Early stopping triggered!")
        break



Epoch 1/15


100%|██████████████████████████████████████████████████████████████████████████| 16946/16946 [1:33:23<00:00,  3.02it/s]


Train Loss: 0.5713
Train Accuracy: 0.7256
Val Accuracy: 0.7267
Val Precision: 0.6788
Val Recall: 0.8608
Val F1: 0.7591
Best model saved!

Epoch 2/15


100%|██████████████████████████████████████████████████████████████████████████| 16946/16946 [1:30:56<00:00,  3.11it/s]


Train Loss: 0.5135
Train Accuracy: 0.7554
Val Accuracy: 0.7433
Val Precision: 0.6938
Val Recall: 0.8713
Val F1: 0.7725
Best model saved!

Epoch 3/15


...

Train Loss: 0.0154
Train Accuracy: 0.9940
Val Accuracy: 0.9846
Val Precision: 0.9779
Val Recall: 0.9915
Val F1: 0.9847
EarlyStopping Counter: 2/5

Epoch 14/15


100%|████████████████████████████████████████████| 16946/16946 [2:54:23<00:00,  1.62it/s]


Train Loss: 0.0146
Train Accuracy: 0.9945
Val Accuracy: 0.9841
Val Precision: 0.9873
Val Recall: 0.9807
Val F1: 0.9840
EarlyStopping Counter: 3/5

Epoch 15/15


100%|████████████████████████████████████████████| 16946/16946 [2:54:31<00:00,  1.62it/s]


Train Loss: 0.0135
Train Accuracy: 0.9949
Val Accuracy: 0.9847
Val Precision: 0.9816
Val Recall: 0.9880
Val F1: 0.9848
EarlyStopping Counter: 4/5


In [12]:
print("\nLoading best model for testing...")
model.load_state_dict(torch.load("best_xception_384.pth"))

test_acc, test_prec, test_rec, test_f1 = evaluate(test_loader)

print("\n==== FINAL TEST RESULTS ====")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall: {test_rec:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")


Loading best model for testing...


C:\Users\abida\AppData\Local\Temp\ipykernel_1992\1747790237.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_xception_384.pth"))



==== FINAL TEST RESULTS ====
Test Accuracy: 0.9840
Test Precision: 0.9825
Test Recall: 0.9855
Test F1 Score: 0.9840
